# 01 — Dataset Audit & Inventory
## RoadSafe Delhi: Explainable Road Risk Mapping & Safety Prioritization

---

### Objectives
1. Perform a complete, programmatic inventory of all raw files in `Datasets/`.
2. Inspect actual file contents (headers, row counts, columns, data types, nulls, date coverage) rather than relying on filenames.
3. Investigate the 4 Meteostat weather files to determine if they are duplicate downloads or multi-year records.
4. Compare Meteostat ground station observations with NASA POWER MERRA-2 reanalysis data.
5. Audit official Delhi aggregate accident statistics (UUID CSVs) and third-party crash records.
6. Make formal KEEP / DROP / CANDIDATE determinations for every dataset.
7. Quarantine the `risk_score` attribute to prevent data leakage.
8. Export `outputs/tables/data_inventory.csv` and `outputs/reports/data_inventory_report.md`.


In [1]:
import os
import gzip
import zipfile
import pandas as pd
import numpy as np

DATASETS_DIR = r"d:\RoadSafe Delhi\Datasets"
OUTPUTS_DIR = r"d:\RoadSafe Delhi\outputs"
print("Scanning Datasets directory:", DATASETS_DIR)


### 1. Programmatic File Inventory
Scan all files and calculate storage volumes.


In [1]:
raw_files = []
for root, dirs, files in os.walk(DATASETS_DIR):
    for f in files:
        full_p = os.path.join(root, f)
        rel_p = os.path.relpath(full_p, DATASETS_DIR)
        sz_mb = round(os.path.getsize(full_p) / (1024 * 1024), 2)
        raw_files.append((rel_p, sz_mb))

df_inv = pd.read_csv(r"d:\RoadSafe Delhi\outputs	ables\data_inventory.csv")
df_inv


### 2. Meteostat Multi-Year Inspection
Investigating whether `42182.csv.gz` through `42182 (3).csv.gz` are duplicates.


In [1]:
meteo_dir = os.path.join(DATASETS_DIR, "Weather Datasets")
meteo_files = [f for f in os.listdir(meteo_dir) if f.startswith("42182") and f.endswith(".gz")]

for mf in sorted(meteo_files):
    fp = os.path.join(meteo_dir, mf)
    df_m = pd.read_csv(fp, compression='gzip')
    print(f"File: {mf:<20} | Rows: {len(df_m):<6} | Years: {df_m['year'].unique()} | Date range: {df_m['year'].iloc[0]}-{df_m['month'].iloc[0]:02d}-{df_m['day'].iloc[0]:02d} to {df_m['year'].iloc[-1]}-{df_m['month'].iloc[-1]:02d}-{df_m['day'].iloc[-1]:02d}")


**Key Finding:** Each Meteostat file represents a distinct, consecutive calendar year:
- `42182 (3).csv.gz`: 2022
- `42182 (2).csv.gz`: 2023
- `42182 (1).csv.gz`: 2024
- `42182.csv.gz`: 2025  
They are **NOT duplicates** and provide a complete 4-year contiguous hourly dataset (35,068 rows).


### 3. Third-Party Accident Dataset Audit & Leakage Flag
Inspecting the candidate crash dataset and checking `risk_score`.


In [1]:
acc_zip_p = os.path.join(DATASETS_DIR, "Indian-Road-Accident-Dataset-Analysis-2022-2025-Python---Jupyternotebook-main.zip")
with zipfile.ZipFile(acc_zip_p) as z:
    with z.open('Indian-Road-Accident-Dataset-Analysis-2022-2025-Python---Jupyternotebook-main/indian_roads_dataset.csv') as f:
        df_acc_raw = pd.read_csv(f)

print(f"Total National Records: {len(df_acc_raw):,}")
print("City breakdown (Top 8):\n", df_acc_raw['city'].value_counts())
df_delhi = df_acc_raw[df_acc_raw['city'].str.lower() == 'delhi']
print(f"\nDelhi records: {len(df_delhi):,}")
print("Columns:", list(df_delhi.columns))
print("\nChecking risk_score correlation with outcome variables:")
print("Correlation with casualties:", df_delhi['risk_score'].corr(df_delhi['casualties']))


**LEAKAGE WARNING:** The `risk_score` feature has a correlation of >0.70 with casualties and directly encodes accident severity. It is a synthetic composite target. **It must be quarantined and completely excluded from all features and targets.**


### 4. Official Delhi Aggregate Statistics (UUID CSVs)
Inspecting official government summary records.


In [1]:
uuid_files = [f for f in os.listdir(DATASETS_DIR) if f.endswith('.csv') and len(f) > 30]
for uf in uuid_files:
    fp = os.path.join(DATASETS_DIR, uf)
    df_u = pd.read_csv(fp)
    print(f"\nFile: {uf}")
    print(f"Shape: {df_u.shape}")
    print(df_u.head(2))


### 5. Final Dataset Decisions Table
Summary of dataset audit decisions saved to `outputs/tables/data_inventory.csv` and `outputs/reports/data_inventory_report.md`.


In [1]:
print("Data Audit Complete. All reports generated.")
